# IEEE-CIS Fraud Detection — Inference

This notebook produces the Kaggle submission file. It does **not** do any preprocessing. The chosen model lives in the MLflow Model Registry as a full sklearn `Pipeline` that includes cleaning + feature engineering + encoding + the trained estimator. We just:

1. Load the data straight from `/kaggle/input/.../test_*.csv`.
2. Pull the registered Pipeline from the Model Registry by name + stage.
3. Call `predict_proba` on the **raw** test DataFrame.
4. Write `submission.csv`.

> The assignment requires the inference path to call `predict` on un-preprocessed test data. If this notebook ever needs to do its own cleaning/FE, that's a sign the training pipeline is incomplete.

## What model is loaded?

By default we load `models:/<MODEL_NAME>/<STAGE>` where stage is **`Production`** for the final winning architecture, or **`Staging`** for in-progress candidates. The variable `MODEL_NAME` near the top is the only thing you need to change to switch which model produces the submission.

# 1. Setup

In [ ]:
# Same setup boilerplate as the model_experiment_*.ipynb notebooks.
# Auto-detects Kaggle vs local Jupyter; resolves the `src/` package via
# (1) attached Kaggle dataset, (2) git clone of REPO_URL, or (3) local `..`.
import os, sys, subprocess, shutil

REPO_URL = "https://github.com/<YOUR_GH_USERNAME>/<YOUR_REPO_NAME>.git"

ON_KAGGLE = os.path.exists("/kaggle/input")
SRC_FOUND = None

if ON_KAGGLE:
    for _d in os.listdir("/kaggle/input"):
        _candidate = f"/kaggle/input/{_d}"
        if os.path.isdir(os.path.join(_candidate, "src")):
            SRC_FOUND = _candidate
            break
    if SRC_FOUND is None:
        REPO_DIR = "/kaggle/working/repo"
        if os.path.isdir(REPO_DIR):
            shutil.rmtree(REPO_DIR)
        subprocess.check_call(["git", "clone", "-q", REPO_URL, REPO_DIR])
        SRC_FOUND = REPO_DIR
    sys.path.insert(0, SRC_FOUND)

    for _pkg in ("mlflow==2.10.2", "dagshub", "xgboost"):
        try:
            __import__(_pkg.split("==")[0].replace("-", "_"))
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _pkg])

    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    for _k in ("MLFLOW_TRACKING_URI", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
        os.environ[_k] = _s.get_secret(_k)
    print("src/ found at :", SRC_FOUND)
else:
    sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn

from src.data import load_test, load_sample_submission
from src.mlflow_utils import setup_mlflow

setup_mlflow()  # no experiment_name -- inference doesn't create runs
print("Tracking URI:", mlflow.get_tracking_uri())

# 2. Pick the registered model

`MODEL_NAME` and `MODEL_STAGE` are the only knobs.

| If you want to submit | Set `MODEL_NAME` to | Set `MODEL_STAGE` to |
|---|---|---|
| The final winning model | `XGBoost_Fraud_Pipeline` (or whichever model you promoted to Production) | `Production` |
| A specific staging candidate | the candidate's name | `Staging` |
| A specific version | any name | `<int>` (e.g. `"3"`) |

In [ ]:
MODEL_NAME  = "XGBoost_Fraud_Pipeline"
MODEL_STAGE = "1"                 # registered version number

model_uri = f"models:/{MODEL_NAME}/{MODEL_STAGE}"
print(f"Loading: {model_uri}")

pipeline = mlflow.sklearn.load_model(model_uri)
print(f"Loaded model class: {type(pipeline).__name__}")
print(pipeline)

# 3. Load raw test data

No preprocessing here. The Pipeline contains it all.

In [ ]:
test = load_test()
print(f"Test shape: {test.shape}")
test.head(2)

# 4. Predict and write submission

The pipeline's `predict_proba` returns a (n, 2) matrix; we keep column 1 (probability of fraud) since the competition metric is ROC-AUC and Kaggle expects probabilities.

In [ ]:
probs = pipeline.predict_proba(test)[:, 1]
print(f"Predictions: shape={probs.shape}  min={probs.min():.4f}  max={probs.max():.4f}  mean={probs.mean():.4f}")

In [ ]:
submission = pd.DataFrame({
    "TransactionID": test["TransactionID"].values,
    "isFraud":       probs,
})

# Sanity check: must align with sample_submission row count and column order
ss = load_sample_submission()
assert len(submission) == len(ss), f"row count mismatch: {len(submission)} vs {len(ss)}"
assert list(submission.columns) == list(ss.columns), "column order mismatch"

OUT_PATH = "/kaggle/working/submission.csv" if ON_KAGGLE else "submission.csv"
submission.to_csv(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH}  ({len(submission):,} rows)")
submission.head()

# 5. Submit to Kaggle

Two options:

1. **From the Kaggle UI**: after this notebook commits, go to the competition page → **Submit Predictions** → choose `submission.csv` from this notebook's output. Kaggle will score it on the public leaderboard.

2. **Via the Kaggle API** (run locally):

   ```bash
   kaggle competitions submit \
       -c ieee-fraud-detection \
       -f submission.csv \
       -m "model: <MODEL_NAME> v<version>"
   ```

After submission, paste the public LB ROC-AUC into the README under **Best Model Results**.